# tensor zeros init — procedural drill

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-zeros-init`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five allocation patterns that ramp from `torch.zeros(n)` → multi-axis shape → `zeros_like` → dtype-long index buffer → allocate-then-scatter for the canonical Ray Tracing per-ray output-buffer pattern. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Core array literacy` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `tensor-zeros-init`**, which bridges to the bank subtopic `Numpy: Core array literacy` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-zeros-init"
DD_SUBTOPIC = "Numpy: Core array literacy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Tensor allocation — quick refresher

**The four shapes of `zeros`:**
- `t.zeros(n)` — 1-D, shape `(n,)`, default `float32`.
- `t.zeros(b, h, w)` — multi-axis positional args.
- `t.zeros_like(x)` — mirror `x.shape` + `x.dtype` + `x.device`.
- `t.zeros(n, dtype=t.long)` — override dtype for index buffers.

**The accumulator pattern.** Allocate the right-shaped zero buffer first; scatter per-element results into it via indexed assignment. Cleaner and faster than `append`-and-stack.

### Exercise 1 — allocate a 1-D zero vector

> ```yaml
> Difficulty: ⚪⚪⚪⚪⚪
> Bloom level: Remember
> LO: Recall the basic `torch.zeros(n)` allocation call.
> Keywords: torch-zeros, shape, dtype-default
> ```

**KCs targeted:** `zeros-1d-shape`

Implement `ex1_zeros_1d(n)` to return a 1-D tensor of `n` floating-point zeros. Default dtype is `torch.float32`.

Use `torch.zeros(n)`.

In [ ]:
def ex1_zeros_1d(n: int) -> Tensor:
    """Return a 1-D tensor of n floating zeros."""
    raise NotImplementedError()


def _test_ex1():
    out = ex1_zeros_1d(5)
    assert out.shape == (5,), f'expected (5,), got {tuple(out.shape)}'
    assert out.dtype == t.float32, f'default dtype should be float32, got {out.dtype}'
    assert t.all(out == 0), f'expected all zeros, got {out.tolist()}'
    assert ex1_zeros_1d(0).shape == (0,), 'n=0 must still return a 0-element tensor (not error)'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_zeros_1d(n: int) -> Tensor:
    return t.zeros(n)
```

**Default dtype is float32.** `torch.zeros(5)` is shorthand for `torch.zeros(5, dtype=torch.float32)`. To get integers you must pass `dtype=` explicitly — see Exercise 4.
</details>

### Exercise 2 — allocate a 3-D zero tensor

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply positional shape arguments to allocate a multi-axis zero tensor.
> Keywords: multi-axis, shape-args, batched-buffer
> ```

**KCs targeted:** `zeros-multi-axis-shape`

Implement `ex2_zeros_3d(b, h, w)` to return a `(b, h, w)` tensor of floating zeros — the kind of buffer you'd allocate for a batch of rendered images.

Either `torch.zeros(b, h, w)` (positional) or `torch.zeros((b, h, w))` (tuple) works.

In [ ]:
def ex2_zeros_3d(b: int, h: int, w: int) -> Tensor:
    """Return a (b, h, w) zero tensor."""
    raise NotImplementedError()


def _test_ex2():
    out = ex2_zeros_3d(2, 3, 4)
    assert out.shape == (2, 3, 4), f'expected (2,3,4), got {tuple(out.shape)}'
    assert out.dtype == t.float32, f'expected float32, got {out.dtype}'
    assert t.all(out == 0), 'expected all zeros'
    # Singleton dims must also work.
    assert ex2_zeros_3d(1, 1, 1).shape == (1, 1, 1), 'singleton dims must be preserved'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_zeros_3d(b: int, h: int, w: int) -> Tensor:
    return t.zeros(b, h, w)
```

**Positional vs tuple.** Both `t.zeros(b, h, w)` and `t.zeros((b, h, w))` produce the same tensor. The positional form is idiomatic when shape is known at write-time; the tuple form is useful when you have a shape computed dynamically (e.g. `t.zeros(x.shape)`).
</details>

### Exercise 3 — zeros_like — mirror an input's shape and dtype

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `torch.zeros_like` to allocate a fresh zero buffer matching the input's shape AND dtype.
> Keywords: zeros-like, shape-mirror, dtype-mirror
> ```

**KCs targeted:** `zeros-like-mirrors-input`

Implement `ex3_zeros_like(x)` to return a fresh zero tensor with the same shape, dtype, and device as `x`.

Use `torch.zeros_like(x)`. Critically: the result must NOT be a view or alias of `x` — writing to the output must not change `x`.

In [ ]:
def ex3_zeros_like(x: Tensor) -> Tensor:
    """Return a fresh zero buffer mirroring x.shape and x.dtype."""
    raise NotImplementedError()


def _test_ex3():
    x_int = t.tensor([[1, 2, 3], [4, 5, 6]], dtype=t.int64)
    out = ex3_zeros_like(x_int)
    assert out.shape == x_int.shape, f'shape mismatch: {out.shape} vs {x_int.shape}'
    assert out.dtype == t.int64, f'dtype must be mirrored: got {out.dtype}'
    assert t.all(out == 0), 'must be all zeros'
    # Aliasing check — writing to out must not mutate x.
    out[0, 0] = 99
    assert x_int[0, 0].item() == 1, 'zeros_like must be a FRESH tensor, not a view'

    x_float = t.randn(4, 5)
    out_f = ex3_zeros_like(x_float)
    assert out_f.shape == x_float.shape
    assert out_f.dtype == t.float32, 'float32 input → float32 output'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_zeros_like(x: Tensor) -> Tensor:
    return t.zeros_like(x)
```

**Why prefer `zeros_like(x)` over `zeros(x.shape)`?**
- `zeros(x.shape)` only copies the shape — the dtype reverts to float32 and the device reverts to CPU. If `x` is a `int64` GPU tensor, your buffer ends up float32 on CPU — silent breakage the moment you try to use it as indices or do an op against `x`.
- `zeros_like(x)` mirrors shape + dtype + device. Always the right call when allocating a per-input accumulator.
</details>

### Exercise 4 — integer index buffer with dtype=long

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `dtype=torch.long` to allocate an integer buffer suitable for use as indices.
> Keywords: dtype-long, index-buffer, gather-ready
> ```

**KCs targeted:** `zeros-dtype-control`

Implement `ex4_index_buffer(n)` to return a 1-D zero tensor of length `n` with dtype `torch.long` (= int64). Integer dtype is required because PyTorch's `gather` / `index_select` / advanced indexing reject float indices.

Use `torch.zeros(n, dtype=torch.long)`.

In [ ]:
def ex4_index_buffer(n: int) -> Tensor:
    """Return a zero index buffer of length n, dtype=long."""
    raise NotImplementedError()


def _test_ex4():
    idx = ex4_index_buffer(4)
    assert idx.shape == (4,), f'expected (4,), got {tuple(idx.shape)}'
    assert idx.dtype == t.long, f'expected dtype long (int64), got {idx.dtype}'
    assert t.all(idx == 0), 'expected all zeros'
    # Critical functional check — must be usable as indices into another tensor.
    source = t.tensor([10.0, 20.0, 30.0])
    gathered = source[idx]  # all zeros → picks element 0 four times.
    assert t.allclose(gathered, t.tensor([10.0, 10.0, 10.0, 10.0])), 'index buffer must work for advanced indexing'
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_index_buffer(n: int) -> Tensor:
    return t.zeros(n, dtype=t.long)
```

**Why `long` and not `int`?** PyTorch's advanced-indexing path requires `int64` (`torch.long`). `torch.int32` works for some ops but fails on `gather` and on CPU advanced indexing — a common confusing footgun. Default to `long` for any tensor that will hold indices.
</details>

### Exercise 5 — allocate output buffer, then paint hits

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize shape arg + dtype default + indexed assignment to scatter per-ray hit colors into an output buffer.
> Keywords: accumulator, indexed-assign, ray-tracing, multi-kc
> ```

**KCs targeted:** `zeros-multi-axis-shape`, `zeros-dtype-control`, `zeros-allocate-then-fill`

Implement `ex5_paint_hits(num_rays, hit_indices, hit_colors)`. The canonical Ray Tracing output-buffer pattern:

1. Allocate a `(num_rays, 3)` zero buffer (float32 by default — perfect for RGB colors in `[0, 1]`).
2. For each `k`, write `hit_colors[k]` into row `hit_indices[k]`.
3. Rays not in `hit_indices` stay `[0, 0, 0]` (black — no hit).

Inputs:
- `num_rays`: int.
- `hit_indices`: 1-D long tensor, shape `(K,)`, values in `[0, num_rays)`.
- `hit_colors`: 2-D float tensor, shape `(K, 3)`.

Output: `(num_rays, 3)` float32 tensor.

Hint: `out[hit_indices] = hit_colors` does the scatter in one shot.

> ⚠️ **Integrative exercise.** This combines 3 KCs (shape allocation, default dtype, indexed assignment); empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step up vs Exercises 1-4.

In [ ]:
def ex5_paint_hits(num_rays: int, hit_indices: Tensor, hit_colors: Tensor) -> Tensor:
    """Allocate (num_rays, 3) zero buffer; write hit_colors at hit_indices."""
    raise NotImplementedError()


def _test_ex5():
    hit_indices = t.tensor([0, 2, 3], dtype=t.long)
    hit_colors = t.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
    out = ex5_paint_hits(5, hit_indices, hit_colors)
    assert out.shape == (5, 3), f'expected (5, 3), got {tuple(out.shape)}'
    assert out.dtype == t.float32, f'expected float32, got {out.dtype}'
    expected = t.tensor([
        [1.0, 0.0, 0.0],  # ray 0 — red hit
        [0.0, 0.0, 0.0],  # ray 1 — no hit, stays zero
        [0.0, 1.0, 0.0],  # ray 2 — green hit
        [0.0, 0.0, 1.0],  # ray 3 — blue hit
        [0.0, 0.0, 0.0],  # ray 4 — no hit, stays zero
    ])
    assert t.allclose(out, expected), f'value mismatch:\n{out}\nvs\n{expected}'
    # Edge case — no hits at all.
    empty_idx = t.zeros(0, dtype=t.long)
    empty_col = t.zeros(0, 3)
    out_empty = ex5_paint_hits(3, empty_idx, empty_col)
    assert out_empty.shape == (3, 3) and t.all(out_empty == 0), 'no-hits case must return all-zero buffer'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_paint_hits(num_rays: int, hit_indices: Tensor, hit_colors: Tensor) -> Tensor:
    out = t.zeros(num_rays, 3)
    out[hit_indices] = hit_colors
    return out
```

**Why this pattern matters.** Every per-ray Ray Tracing computation uses this shape: allocate a `(num_rays, ...)` output buffer with the right dtype, compute the mask of which rays did something, scatter the per-hit values back in. Rays that don't hit anything keep the default fill (zero / -inf / NaN sentinel depending on the use).

**Why the indexed-assign works.** `out[hit_indices] = hit_colors` uses advanced indexing: PyTorch evaluates `hit_indices` as a list of row positions and writes the matching row from `hit_colors` into each. Requires `hit_indices.dtype == long` — see Exercise 4.
</details>

## Done

Run the cell below to report your progress to Delta Drills. The beacon fires only if all 5 exercises passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1', 'ex2', 'ex3', 'ex4', 'ex5'}

def _dd_feedback_level(num_passed: int) -> str:
    """Map exercise-pass count → arena-rating feedback enum."""
    if num_passed == 5: return 'not_much'
    if num_passed >= 3: return 'somewhat'
    return 'a_lot'

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {len(missing)} exercises still failing: {sorted(missing)}.")
        print("[Delta Drills] not reporting until all 5 pass.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}',
        'subtopics': [DD_SUBTOPIC],
        'feedback': _dd_feedback_level(len(_dd_passed)),
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()